<a href="https://colab.research.google.com/github/AkMebo/ML_Lesson1_flyrankAI/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AkMebo/ML_Lesson1_flyrankAI/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

Clustering - “this is unsupervised, so I judge it by separation, stability and usefulness not accuracy.”

The model will find groups of pages that behave alike. I will learn each groups typical number and which action fits.

I consider clustering because there is no label to predict.

My question is “what recurring performance types exist across the content inventory?” There is no label column that says which page is a champion or a hidden gem, so classification has nothing to learn from. I’m not ordering pages (ranking) or giving each one a single number (scoring), because a page’s behavior has several sides: demand, visibility, trend, and freshness.





In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

Target: none (clustering is unsupervised). The model learns groups from the features only and is not trained on any label.

I do care about content with decreasing visibility, so I will use it as a proxy check, not a target. After clustering, I will measure how many pages in each cluster show falling visibility, defined by a rule using e.g. impressions_last_30d, impressions_prev_30d. That label comes from a defined rule, not an observed outcome, and it is never fed to the model.


We can also identify certain factors that are prevalent in the content with decreasing visibility:
1. Primary proxy - trend_direction (down / stable / up / new / flat)
2. Secondary - clicks_last_30d, clicks_prev_30d



In [2]:
!git clone https://github.com/AkMebo/ML_Lesson1_flyrankAI
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


Cloning into 'ML_Lesson1_flyrankAI'...
remote: Enumerating objects: 142, done.
remote: Counting objects: 100% (142/142), done.
remote: Compressing objects: 100% (98/98), done.
remote: Total 142 (delta 48), reused 95 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (142/142), 1.88 MiB | 6.66 MiB/s, done.
Resolving deltas: 100% (48/48), done.


In [ ]:
import pandas as pd, numpy as nm

content_df = pd.read_csv('ML_Lesson1_flyrankAI/data/raw/content_refresh_anonymized.csv')

s = content_df['impressions_90d']
print("rows:", len(s), "| missing:", s.isna().sum())
print(s.describe().round(1))

rows: 30000 | missing: 0
count     30000.0
mean       5200.4
std       16838.0
min           1.0
25%          81.0
50%         731.0
75%        3615.2
max      517715.0
Name: impressions_90d, dtype: float64


In [3]:
import pandas as pd, numpy as nm

content_df = pd.read_csv('ML_Lesson1_flyrankAI/data/raw/content_refresh_anonymized.csv')
df = content_df.copy()
print(df['trend_direction'].value_counts())          # check how many 'flat' and 'new'
df['declining_visibility'] = df['trend_direction'].apply(lambda x: 1 if x == 'down' else 0)

# Optional volume floor so tiny pages don't distort it
floor = 100    # your choice; state it in the notebook
big = df['impressions_90d'] >= floor
print("declining rate, all pages:", df['declining_visibility'].mean().round(3))
print("declining rate, >=", floor, "impressions:", df.loc[big, 'declining_visibility'].mean().round(3), "| rows:", big.sum())

print(df['trend_direction'].value_counts())
print(df['trend_direction'].value_counts(normalize=True).round(3))

trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64
declining rate, all pages: 0.542
declining rate, >= 100 impressions: 0.598 | rows: 22006
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64
trend_direction
down      0.542
stable    0.199
up        0.146
new       0.075
flat      0.038
Name: proportion, dtype: float64


## 3. Success metric

*One metric (is a number computed from the results) you can defend. What number means 'good'?*

Success metric: cluster stability, measured by the Adjusted Rand Index (ARI).

I will re-run the clustering with different random seeds and on random subsamples and measure how often pages that were grouped together stay together.
fixed before I look at results
* Good = mean ARI ≥ 0.80
* 0.60 to 0.80 means soft groups;
* < 0.60 means I will not present them as archetypes.

Supporting evidence:
1. groups are distinct, shown by per-cluster medians and a silhouette score;
2. each kept group is useful, meaning it maps to a different action;
3. clustering beats a simple baseline (impression-tier grouping) on the same metric.

This is unsupervised, so I do not use accuracy. Stable groups show the structure is not an accident of the seed. They do not prove the groups are true categories.  


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

One row = one page (one content item), identified by content_id

Evidence from the data dictionary:

* The file says “One row per content item (page).”
* content_id is described as unique per row.
* 30,000 rows is 30,000 pages, across 32 clients.

In [ ]:
print("rows:", len(df), "| unique pages:", df['content_id'].nunique(), "| one row per page:", len(df) == df['content_id'].nunique())
print("clients:", df['client_id'].nunique())


# lane's slice: id for tracking, features to cluster on, trend held out for the check
id_cols      = ['content_id', 'client_id'] #for tracking and splitting only
cluster_cols = ['impressions_90d', 'clicks_90d', 'ctr', 'avg_position', 'sessions_90d', 'engagement_rate', 'scroll_rate','days_with_impressions',
                'content_age_days', 'days_since_last_update']
check_cols   = ['trend_direction', 'trend_pct']          # held out, never fed to the clustering

lane_df = df[id_cols + cluster_cols + check_cols]
lane_df.head(10)

rows: 30000 | unique pages: 30000 | one row per page: True
clients: 32


,content_id,client_id,impressions_90d,clicks_90d,ctr,avg_position,sessions_90d,engagement_rate,scroll_rate,days_with_impressions,content_age_days,days_since_last_update,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,3803,29,0.76,10.6,17,5.88,4.55,88,187,20,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,15320,7,0.05,20.3,9,0.00,10.00,88,445,25,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,12581,11,0.09,36.5,11,0.00,28.57,88,141,20,down,-60.9
3,content_331d6c4de07b,client_19581e27de,11751,58,0.49,6.2,78,1.28,3.45,88,463,22,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,19140,24,0.13,44.0,145,0.00,24.29,88,263,14,down,-34.7
5,content_d4084a4bc775,client_f369cb89fc,3970,1,0.03,8.5,5,0.00,25.00,88,147,20,down,-38.9
6,content_9a34b442b552,client_8722616204,20,0,0.00,7.0,1,0.00,0.00,13,90,20,down,-92.3
7,content_a63219c6e95a,client_19581e27de,1724,1,0.06,21.2,28,3.57,7.14,88,445,22,stable,0.6
8,content_5e6c160719bc,client_6208ef0f77,32574,29,0.09,46.0,68,5.88,6.25,83,90,20,down,-58.8
9,content_c27558df2b0c,client_19581e27de,1240,2,0.16,4.9,3,0.00,0.00,88,257,104,down,-29.2


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

1. Too many dimensions - A page’s type depends on several columns together (visibility, clicks, engagement, consistency, age). A rule on one column misses combinations, like “visible but not engaging.
2. Heavy skew - Mean 5,200 vs. median 730 means a few giant pages dominate. Any cutoff you pick is arbitrary: why 3,000, not 2,500?
3. No sharp boundaries - Pages shade smoothly from weak to strong. There’s no gap where a cutoff naturally falls
4. Meaning depends on context. The same traffic means something different for a young page and an old one, or a recently updated one and a stale one.
5. Zeros and blanks are not always real values. avg_position = 0 means “no data,” so a cutoff on that column would misplace those pages.

In [ ]:
cols = ['impressions_90d','clicks_90d','sessions_90d','engagement_rate',
        'days_with_impressions','content_age_days','days_since_last_update']

# 1. Dimensions: do columns move together or independently? (rank correlation handles skew)
print(df[cols].corr(method='spearman').round(2))

# 2. Skew
print(df[cols].skew().round(1))

# 4. Same visibility, different engagement? (inside one impression tier)
mid = df[df['impression_tier'].isin(['moderate','good'])]
print(mid['engagement_rate'].describe(percentiles=[.1,.5,.9]).round(1))
print(df.groupby('age_tier')['clicks_90d'].median())

# 5. Fake zeros
print("avg_position == 0:", (df['avg_position'] == 0).sum())


                        impressions_90d  clicks_90d  sessions_90d  \
impressions_90d                    1.00        0.83          0.75   
clicks_90d                         0.83        1.00          0.70   
sessions_90d                       0.75        0.70          1.00   
engagement_rate                    0.49        0.54          0.50   
days_with_impressions              0.92        0.74          0.68   
content_age_days                   0.01       -0.05          0.00   
days_since_last_update             0.25        0.15          0.26   

                        engagement_rate  days_with_impressions  \
impressions_90d                    0.49                   0.92   
clicks_90d                         0.54                   0.74   
sessions_90d                       0.50                   0.68   
engagement_rate                    1.00                   0.42   
days_with_impressions              0.42                   1.00   
content_age_days                   0.02            

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.